<a href="https://colab.research.google.com/github/syhennie/data-quality-process/blob/main/features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Настройка окружения

In [1]:
%pip install numpy gensim pandas scipy sdv matplotlib graphviz faker

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sdv
import pathlib

In [3]:
df = pd.read_csv(pathlib.Path('../test.csv'))
df = df.iloc[:, 1:]
df.head()

,article,highlights
0,Ever noticed how plane seats appear to be gett...,Experts question if packed out planes are put...
1,A drunk teenage boy had to be rescued by secur...,Drunk teenage boy climbed into lion enclosure ...
2,Dougie Freedman is on the verge of agreeing a ...,Nottingham Forest are close to extending Dougi...
3,Liverpool target Neto is also wanted by PSG an...,Fiorentina goalkeeper Neto has been linked wit...
4,Bruce Jenner will break his silence in a two-h...,"Tell-all interview with the reality TV star, 6..."


In [4]:
from gensim.models import Word2Vec, FastText
from gensim.utils import simple_preprocess

model = FastText.load("../fasttext_lee_background")

MAX_TOKENS_PER_ENTRY = 10
VECTOR_SIZE = 64

def vectorise_entries(entries):
    features = []
    for entry in entries:
        tokens = simple_preprocess(entry)
        vectors = [model.wv[token] for token in tokens]
        length = min(len(vectors), MAX_TOKENS_PER_ENTRY)
        trimmed_vectors = vectors[:length]
        if length < MAX_TOKENS_PER_ENTRY:
            padding_vectors = [np.zeros(VECTOR_SIZE) for _ in range(MAX_TOKENS_PER_ENTRY - length)]
            trimmed_vectors += padding_vectors
        features.append(np.concatenate(trimmed_vectors))
    return np.array(features)

In [5]:
def decode_row(row):
    entry = ""
    for i in range(MAX_TOKENS_PER_ENTRY):
        vectorised_token = row[i * VECTOR_SIZE:(i + 1) * VECTOR_SIZE]
        most_similar_words = model.wv.similar_by_vector(vectorised_token, topn=5)
        for word, _ in most_similar_words:
            if str(word).isalnum():
                entry += f" {word}"
    return entry

In [6]:
import torch
print(torch.cuda.is_available())
num_gpus = torch.cuda.device_count()
for i in range(num_gpus):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")
torch.cuda.set_device(0)

True
Device 0: NVIDIA GeForce RTX 5070 Ti


In [7]:
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer


df_sample = df.sample(1000)
df_result = pd.DataFrame()

for column in df_sample.columns:
    entries = [x for x in df_sample[column]]
    vectorised_entries = vectorise_entries(entries)

    column_df = pd.DataFrame(vectorised_entries)
    metadata = Metadata.detect_from_dataframe(column_df)
    synthesizer = CTGANSynthesizer(metadata)
    synthesizer.fit(column_df)

    synthetic_data = synthesizer.sample(num_rows=1000).values
    result = np.apply_along_axis(decode_row, 1, synthetic_data)
    df_result[column] = result

df_result
    

c:\Users\stepa\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
0                      11
1                      11
2                      11
3                      11
4                      11
5                      11
6                      11
7                      11
8                      11
9                      11
10                     11
11                     11
12                     11
13                     11
14                     11
15                     11
16                     11
17                     11
18                     11
19                     11
20                     11
21                     11
22                     11
23                     11
24                     11
25                     11
26                     11
27                     11
28                     11
29                     11
30                     11


c:\Users\stepa\AppData\Local\Programs\Python\Python311\Lib\site-packages\ctgan\synthesizers\_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
c:\Users\stepa\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
0                      11
1                      11
2                      11
3                      11
4                      11
5                      11
6                      11
7                      11
8                      11
9                      11
10                     11
11                     11
12                     11
13                     11
14                     11
15                     11
16                     11
17                     11
18                     11
19                     11
20                     11
21                     11
22                     11
23                     11
24                     11
25                     11
26                     11
27                     11
28                     11
29                     11
30                     11


c:\Users\stepa\AppData\Local\Programs\Python\Python311\Lib\site-packages\ctgan\synthesizers\_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


,article,highlights
0,24 48 26 80 AFP ANZ HIV 1 25 blaze drug am Zi...,23 80 At W 24 AFP Up 15 SES HIV drug W 25 40 ...
1,25 150 my If Fatah 26 24 48 If Up 2 On UN 150...,Mt 13 21 12 150 1 26 14 drug Assa 15 Up 12 HI...
2,blaze 21 Sarah gun Nauru 11 25 Up No 48 No UN...,24 SES ANZ embassy W 13 25 18 Ms As 40 15 14 ...
3,80 21 On If Nauru 150 Fatah Doug If 48 Mt AFP...,11 20 Fatah Road 12 As drug Assa 5 SES Mt Up ...
4,AFP Up She keep His Up As 2 W 25 Up 1 Nauru 2...,80 24 18 2 HIH AFP 14 W 26 Bonn Blue my 150 S...
...,...,...
995,On No HIV 11 W UN W No At 11 23 500 400 Europ...,HIV Fatah drug oil Two 5 80 40 HIV drug 80 26...
996,26 48 Nauru Bonn school Up HIV 48 24 ANZ Up M...,26 1 W ANZ At 14 Lee ANZ Africa 48 15 150 Dr ...
997,gun 23 40 24 She 24 W 40 blaze At 23 News 14 ...,Up 14 12 1 23 Bonn Nauru 14 18 14 Kabul 12 Tw...
998,If 150 various boy HIV SES 24 W Ms Hewitt 20 ...,13 15 HIV 14 26 Up 25 As 150 23 21 500 13 Mt ...


In [8]:
def get_vectorised_entries(entries):
    features = []
    for entry in entries:
        tokens = simple_preprocess(entry, min_len=1)
        vectors = [model.wv[token] for token in tokens]
        features.append(np.mean(vectors, axis=0))
    return np.array(features)

In [12]:
summary_stats = []
features_data = []

for column in df.columns:
    entries = [x for x in df[column]]
    features = get_vectorised_entries(entries)
    vectorised_entries = vectorise_entries(entries)

    # Features-based metrics
    #
    # mean = np.mean(features, axis=0)
    # standard_deviation = np.std(features, axis=0)
    # median = np.median(features, axis=0)
    # asymmetry = stats.skew(features, axis=0)
    # excess = stats.kurtosis(features, axis=0)

    mean = np.mean(vectorised_entries, axis=0)
    standard_deviation = np.std(vectorised_entries, axis=0)
    median = np.median(vectorised_entries, axis=0)
    asymmetry = stats.skew(vectorised_entries, axis=0)
    excess = stats.kurtosis(vectorised_entries, axis=0)

    summary_stats.append({
        'column': column,
        'overall_mean': np.mean(mean),
        'overall_std': np.mean(standard_deviation),
        'std_of_means': np.std(mean),
        'mean_of_medians': np.mean(median),
        'asymmetry_avg': np.mean(asymmetry),
        'excess_avg': np.mean(excess),
        'n_entries': len(entries),
        'vector_dim': vectorised_entries.shape[1]
    })

    features_data.append({
            'column': column,
            'vectors': features
        })

summary_df = pd.DataFrame(summary_stats)
summary_df

,column,overall_mean,overall_std,std_of_means,mean_of_medians,asymmetry_avg,excess_avg,n_entries,vector_dim
0,article,-0.033408,0.189726,0.417737,-0.031447,-0.064652,-0.125835,11490,640
1,highlights,-0.032024,0.195210,0.403500,-0.028976,-0.088079,0.203277,11490,640


In [13]:
def validate_synthetic_data(original_stats, synthetic_data):
    validation_results = []

    for column in synthetic_data.columns:
        column_stats = original_stats[original_stats['column'] == column]
        entries = [x for x in synthetic_data[column]]
        features = get_vectorised_entries(entries)

        synth_mean = np.mean(features, axis=0)
        synth_std = np.std(features, axis=0)
        synth_skew = stats.skew(features, axis=0)
        synth_kurt = stats.kurtosis(features, axis=0)

        validation_results.append({
            'column': column_stats['column'].tolist()[0],
            'original_mean': column_stats['overall_mean'].tolist()[0],
            'synthetic_mean': np.mean(synth_mean),
            'mean_error': np.abs(np.mean(synth_mean) - column_stats['overall_mean'].tolist()[0]),

            'original_std': column_stats['overall_std'].tolist()[0],
            'synthetic_std': np.mean(synth_std),
            'std_error': np.abs(np.mean(synth_std) - column_stats['overall_std'].tolist()[0]),

            'original_skew': column_stats['asymmetry_avg'].tolist()[0],
            'synthetic_skew': np.mean(synth_skew),
            'skew_error': np.abs(np.mean(synth_skew) - column_stats['asymmetry_avg'].tolist()[0]),

            'original_kurt': column_stats['excess_avg'].tolist()[0],
            'synthetic_kurt': np.mean(synth_kurt),
            'kurt_error': np.abs(np.mean(synth_kurt) - column_stats['excess_avg'].tolist()[0]),

            'n_samples': len(features),
            'vector_dim': features.shape[1]
        })

    return pd.DataFrame(validation_results)

In [15]:
validation_df = validate_synthetic_data(summary_df, df_result)

# validation_df[['column', 'mean_error', 'std_error', 'skew_error', 'kurt_error']]
validation_df[['column', 'original_mean', 'synthetic_mean', 'mean_error', 'original_std', 'synthetic_std', 'std_error', 'original_skew', 'synthetic_skew', 'skew_error', 'original_kurt', 'synthetic_kurt', 'kurt_error']]
# print(validation_df[['column', 'mean_error', 'std_error', 'skew_error', 'kurt_error']])

,column,original_mean,synthetic_mean,mean_error,original_std,synthetic_std,std_error,original_skew,synthetic_skew,skew_error,original_kurt,synthetic_kurt,kurt_error
0,article,-0.033408,-0.019497,0.013911,0.189726,0.038916,0.150809,-0.064652,-0.013004,0.051648,-0.125835,0.053503,0.179337
1,highlights,-0.032024,-0.017962,0.014061,0.195210,0.038512,0.156698,-0.088079,-0.024766,0.063313,0.203277,-0.172453,0.375730
